In [1]:
import numpy as np
from pathlib import Path
import json

# List your previous analysis folders
SESSIONS_TO_CHECK = [
    (r"G:\Kevin\2026-03-04_08-31-33\Record Node 101\experiment1\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4",
     r"G:\Kevin\2026-03-04_08-31-33\Record Node 101\2026-03-04_08-31-33_experiment1_recording1_analysis"),
    (r"G:\Kevin\2026-03-05_09-27-03\Record Node 101\experiment2\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4",
     r"G:\Kevin\2026-03-05_09-27-03\Record Node 101\2026-03-05_09-27-03_experiment2_recording1_analysis"),
    (r"G:\Kevin\2026-03-09_14-29-25\Record Node 114\experiment2\recording1\continuous\Neuropix-PXI-107.ProbeA\kilosort4",
     r"G:\Kevin\2026-03-09_14-29-25\Record Node 114\2026-03-09_14-29-25_experiment2_recording1_analysis"),
]

NP_SAMPLE_RATE = 30000.0

for probe_path_str, analysis_folder_str in SESSIONS_TO_CHECK:
    probe_path = Path(probe_path_str)
    analysis_folder = Path(analysis_folder_str)

    if not probe_path.exists():
        print(f"\nMISSING: {probe_path}")
        continue

    # Load spike times and probe continuous timestamps
    spike_times = np.load(probe_path / "spike_times.npy").flatten()
    probe_cont = probe_path.parent
    ts = np.load(probe_cont / "timestamps.npy")

    np_start_abs = float(ts[0])
    np_end_abs = float(ts[-1])
    recording_duration = np_end_abs - np_start_abs

    spike_min_s = spike_times.min() / NP_SAMPLE_RATE
    spike_max_s = spike_times.max() / NP_SAMPLE_RATE

    # What anchor does the data suggest?
    np_anchor_check = (abs(spike_min_s) < 5.0 and
                       spike_max_s <= recording_duration + 5.0)
    abs_anchor_check = (abs(spike_min_s - np_start_abs) < 5.0 and
                        abs(spike_max_s - np_end_abs) < 5.0)

    print(f"\n=== {Path(probe_path_str).parts[-5]} ===")
    print(f"  NP stream:    {np_start_abs:.3f} → {np_end_abs:.3f} s "
          f"(duration: {recording_duration:.3f} s)")
    print(f"  Spike times:  {spike_min_s:.3f} → {spike_max_s:.3f} s")
    print(f"  NP-anchor match (spikes start near 0): {np_anchor_check}")
    print(f"  ABS-anchor match (spikes start near {np_start_abs:.1f}): {abs_anchor_check}")
    if np_anchor_check and not abs_anchor_check:
        print(f"  → CORRECT: spikes anchored to NP start (offset=0)")
    elif abs_anchor_check and not np_anchor_check:
        print(f"  → BUG: spikes anchored to absolute clock. "
              f"Need offset = {np_start_abs:.6f} s")
    elif np_anchor_check and abs_anchor_check:
        # This happens when np_start_abs is very small
        print(f"  → AMBIGUOUS: np_start_abs is small ({np_start_abs:.3f}). "
              f"Both anchors work equivalently.")
    else:
        print(f"  → UNCLEAR: neither anchor matches.")


=== experiment1 ===
  NP stream:    12.782 → 2460.072 s (duration: 2447.289 s)
  Spike times:  0.000 → 2447.289 s
  NP-anchor match (spikes start near 0): True
  ABS-anchor match (spikes start near 12.8): False
  → CORRECT: spikes anchored to NP start (offset=0)

=== experiment2 ===
  NP stream:    4.325 → 3733.729 s (duration: 3729.404 s)
  Spike times:  0.000 → 3729.404 s
  NP-anchor match (spikes start near 0): True
  ABS-anchor match (spikes start near 4.3): True
  → AMBIGUOUS: np_start_abs is small (4.325). Both anchors work equivalently.

=== experiment2 ===
  NP stream:    171.413 → 2163.615 s (duration: 1992.203 s)
  Spike times:  0.000 → 1991.999 s
  NP-anchor match (spikes start near 0): True
  ABS-anchor match (spikes start near 171.4): False
  → CORRECT: spikes anchored to NP start (offset=0)
